# Training Results Analysis: All 105 Models

**Author**: Glenn Dalbey  
**Date**: 2025-10-17  
**Project**: RSNA 2025 Intracranial Aneurysm Detection

---

## Overview

Comprehensive analysis of training results from 105 models (21 architectures × 5 folds). This represents ~300 GPU-hours of training completed over 2 weeks on 4 GPUs (2× RTX 5090 + RTX 3090 Ti + RTX 3090), all running locally.

### Training Configuration

- **Dataset**: 4,348 CT angiography scans (3,478 train / 870 validation per fold)
- **Input**: 64×64×64 voxel patches, CTA windowed, z-score normalized
- **Loss**: BCEWithLogitsLoss with class weighting (82.3× max weight)
- **Optimizer**: Adam (lr=0.001 baseline, 0.0005 optimized)
- **Scheduler**: CosineAnnealingLR with warmup
- **Early Stopping**: Patience=10-15 epochs based on validation AUC
- **Hardware**: Mixed precision (FP16) on NVIDIA RTX GPUs

### Key Results

- **Best Single Model**: SE-ResNet18 Stable (AUC: 0.8585)
- **Most Stable**: SE-ResNet34 (CV: 0.61%)
- **Fastest Convergence**: MobileNetV4 (~15 epochs)
- **Catastrophic Failures**: All Transformers from scratch (AUC < 0.68)

### Notebook Sections

1. Model Performance Overview (Heatmap)
2. Architecture Family Performance
3. Training Convergence Analysis
4. Model Size vs Performance (Critical)
5. Hyperparameter Impact Analysis
6. Cross-Fold Variance Analysis
7. Failed Experiments Documentation
8. Statistical Significance Testing
9. Key Insights Summary
10. Results Export

In [1]:
# Verify environment and install dependencies if needed
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

# Install plotly if not available
try:
    import plotly
    print(f"Plotly version: {plotly.__version__}")
except ImportError:
    print("Installing plotly...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "kaleido", "-q"])
    print("Plotly installed successfully")

Python executable: /home/yeblad/anaconda3/envs/rsna_kaggle/bin/python3.11
Python version: 3.11.13 (main, Jun  5 2025, 13:12:00) [GCC 11.2.0]
Plotly version: 6.3.1


In [2]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Dict, List, Tuple
import json
import re
from scipy import stats
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (18, 10)
plt.rcParams['font.size'] = 11

# Paths
BASE_DIR = Path('/mnt/raid0/kaggle_rsna_full')
WORKSPACE_DIR = BASE_DIR / 'workspace'
LOGS_DIR = WORKSPACE_DIR / 'logs'
GITHUB_DIR = BASE_DIR / 'rsna_github'

print(f"Workspace directory: {WORKSPACE_DIR}")
print(f"Logs directory: {LOGS_DIR}")
print(f"Logs exist: {LOGS_DIR.exists()}")

Workspace directory: /mnt/raid0/kaggle_rsna_full/workspace
Logs directory: /mnt/raid0/kaggle_rsna_full/workspace/logs
Logs exist: True


## 1. Model Performance Overview

Interactive heatmap of all 105 models ranked by performance.

In [3]:
# Parse all training logs to extract results
def parse_log_file(log_path: Path) -> Dict:
    """Extract best AUC from training log."""
    try:
        with open(log_path, 'r') as f:
            content = f.read()
        
        # Look for best AUC pattern
        auc_pattern = r'best.*?AUC.*?([0-9]\.[0-9]{4})'
        matches = re.findall(auc_pattern, content, re.IGNORECASE)
        
        if matches:
            return {'auc': float(matches[-1]), 'log_file': log_path.name}
        else:
            return None
    except Exception as e:
        return None

# Scan all log files
results = []
if LOGS_DIR.exists():
    log_files = list(LOGS_DIR.glob('*.log'))
    print(f"Found {len(log_files)} log files")
    
    for log_file in log_files:
        result = parse_log_file(log_file)
        if result:
            # Extract metadata from filename
            filename = log_file.stem
            
            # Parse fold number
            fold_match = re.search(r'fold[_]?(\d+)', filename)
            fold = int(fold_match.group(1)) if fold_match else 0
            
            # Parse architecture name
            arch_patterns = [
                r'(seresnet\d+)',
                r'(resnet\d+)',
                r'(densenet\d+)',
                r'(efficientnet[_-]b\d+)',
                r'(mobilenetv\d+)',
                r'(vit[_-]\w+)',
                r'(swin[_-]\w+)',
                r'(convnext[_-]\w+)'
            ]
            
            arch = 'unknown'
            for pattern in arch_patterns:
                match = re.search(pattern, filename.lower())
                if match:
                    arch = match.group(1)
                    break
            
            results.append({
                'architecture': arch,
                'fold': fold,
                'auc': result['auc'],
                'log_file': result['log_file']
            })

df_results = pd.DataFrame(results)
print(f"\nSuccessfully parsed {len(df_results)} training runs")
print(f"Unique architectures: {df_results['architecture'].nunique()}")
print(f"Folds covered: {sorted(df_results['fold'].unique())}")

# Create pivot table for heatmap
if len(df_results) > 0:
    pivot = df_results.pivot_table(values='auc', index='architecture', columns='fold', aggfunc='first')
    pivot = pivot.sort_values(by=list(pivot.columns), ascending=False)
    
    # Add mean and std columns
    pivot['Mean'] = pivot.mean(axis=1)
    pivot['Std'] = pivot.std(axis=1)
    pivot = pivot.sort_values('Mean', ascending=False)
    
    # Create interactive heatmap
    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=['Fold 0', 'Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Mean', 'Std'],
        y=pivot.index,
        colorscale='RdYlGn',
        zmid=0.80,
        text=np.round(pivot.values, 4),
        texttemplate='%{text}',
        textfont={"size": 9},
        colorbar=dict(title="AUC")
    ))
    
    fig.update_layout(
        title='All 105 Models Ranked by Performance (21 Architectures × 5 Folds)',
        xaxis_title='Fold',
        yaxis_title='Architecture',
        height=max(600, len(pivot) * 25),
        font=dict(size=11)
    )
    
    fig.show()
    
    # Summary statistics
    print("\n=== Overall Performance Summary ===")
    print(f"Best model: {pivot['Mean'].idxmax()} (Mean AUC: {pivot['Mean'].max():.4f})")
    print(f"Worst model: {pivot['Mean'].idxmin()} (Mean AUC: {pivot['Mean'].min():.4f})")
    print(f"Performance spread: {(pivot['Mean'].max() - pivot['Mean'].min())*100:.2f} percentage points")
    print(f"\nMost stable: {pivot['Std'].idxmin()} (Std: {pivot['Std'].min():.4f})")
    print(f"Least stable: {pivot['Std'].idxmax()} (Std: {pivot['Std'].max():.4f})")
    
    # Top 10 models
    print("\n=== Top 10 Models (by Mean AUC) ===")
    display(pivot.head(10))
else:
    print("\nWARNING: No training results found in logs directory")
    print("Please ensure log files are present and contain 'best AUC' entries")

Found 288 log files



Successfully parsed 228 training runs
Unique architectures: 54
Folds covered: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]



=== Overall Performance Summary ===
Best model: efficientnet_b0 (Mean AUC: 0.8492)
Worst model: swin_p64 (Mean AUC: 0.5422)
Performance spread: 30.70 percentage points

Most stable: efficientnet_b0 (Std: 0.0000)
Least stable: convnext_large_finetune (Std: 0.1171)

=== Top 10 Models (by Mean AUC) ===


fold,0,1,2,3,4,Mean,Std
architecture,,,,,,,
efficientnet_b0,0.8492,NaN,NaN,NaN,NaN,0.84920,0.000000
mobilenetv3,0.8483,NaN,NaN,NaN,NaN,0.84830,0.000000
mobilenetv4,0.8559,0.8579,0.8413,0.8411,0.8439,0.84802,0.007345
seresnet10,0.8584,0.8465,NaN,NaN,0.8370,0.84730,0.008755
seresnet34,0.8550,0.8498,0.8427,0.8414,0.8473,0.84724,0.004929
seresnet18,0.8545,0.8507,0.8490,0.8490,0.8316,0.84696,0.007938
mobilenetv2,0.8463,NaN,NaN,NaN,NaN,0.84630,0.000000
efficientnet_b2,0.8453,NaN,NaN,NaN,NaN,0.84530,0.000000
seresnet14,0.8457,0.8518,0.8523,0.8382,0.8376,0.84512,0.006340


## 2. Architecture Family Performance

Box plot comparison across architecture families showing distribution and outliers.

In [4]:
# Categorize architectures into families
def categorize_architecture(arch_name: str) -> str:
    arch_lower = arch_name.lower()
    if 'seresnet' in arch_lower:
        return 'SE-ResNet'
    elif 'resnet' in arch_lower:
        return 'ResNet'
    elif 'densenet' in arch_lower:
        return 'DenseNet'
    elif 'efficientnet' in arch_lower:
        return 'EfficientNet'
    elif 'mobilenet' in arch_lower:
        return 'MobileNet'
    elif 'vit' in arch_lower:
        return 'ViT'
    elif 'swin' in arch_lower:
        return 'Swin Transformer'
    elif 'convnext' in arch_lower:
        return 'ConvNeXt'
    else:
        return 'Other'

if len(df_results) > 0:
    df_results['family'] = df_results['architecture'].apply(categorize_architecture)
    
    # Box plot with individual points
    fig = px.box(
        df_results,
        x='family',
        y='auc',
        color='family',
        points='all',
        title='Architecture Family Performance Distribution (All Folds)',
        labels={'auc': 'Validation AUC', 'family': 'Architecture Family'},
        color_discrete_sequence=px.colors.qualitative.Set2,
        height=700
    )
    
    fig.update_layout(
        font=dict(size=12),
        showlegend=False,
        xaxis={'tickangle': -45}
    )
    
    fig.show()
    
    # Family statistics
    print("\n=== Architecture Family Statistics ===")
    family_stats = df_results.groupby('family')['auc'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).sort_values('mean', ascending=False)
    family_stats['range'] = family_stats['max'] - family_stats['min']
    display(family_stats)
    
    # ANOVA test for family differences
    family_groups = [df_results[df_results['family'] == family]['auc'].values 
                     for family in df_results['family'].unique()]
    f_stat, p_value = stats.f_oneway(*family_groups)
    
    print(f"\n=== ANOVA Test ===")
    print(f"F-statistic: {f_stat:.4f}")
    print(f"P-value: {p_value:.4e}")
    print(f"Conclusion: {'Statistically significant' if p_value < 0.05 else 'Not significant'} difference between families")


=== Architecture Family Statistics ===


,count,mean,std,min,max,range
family,,,,,,
MobileNet,8,0.848600,0.006623,0.8411,0.8579,0.0168
SE-ResNet,31,0.846006,0.008751,0.8217,0.8585,0.0368
DenseNet,28,0.809007,0.063290,0.6781,0.8545,0.1764
Other,34,0.754950,0.089062,0.6220,0.8491,0.2271
EfficientNet,26,0.739300,0.087844,0.6217,0.8492,0.2275
ResNet,55,0.738587,0.077660,0.6050,0.8498,0.2448
ConvNeXt,29,0.726610,0.101397,0.5764,0.8540,0.2776
ViT,13,0.701146,0.072548,0.5713,0.8203,0.2490
Swin Transformer,4,0.688150,0.148426,0.5422,0.8177,0.2755



=== ANOVA Test ===
F-statistic: 10.2849
P-value: 3.4341e-12
Conclusion: Statistically significant difference between families


## 3. Training Convergence Analysis

Analyzing training dynamics and convergence speed for top models.

In [5]:
# Parse training curves from logs (epoch-by-epoch)
def parse_training_curve(log_path: Path) -> Dict:
    """Extract training history from log file."""
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
        
        epochs = []
        train_loss = []
        val_loss = []
        val_auc = []
        
        for line in lines:
            # Look for epoch summaries
            epoch_match = re.search(r'Epoch\s+(\d+)', line, re.IGNORECASE)
            if epoch_match:
                current_epoch = int(epoch_match.group(1))
                
                # Extract metrics
                train_loss_match = re.search(r'train.*?loss.*?([0-9]\.[0-9]+)', line, re.IGNORECASE)
                val_loss_match = re.search(r'val.*?loss.*?([0-9]\.[0-9]+)', line, re.IGNORECASE)
                auc_match = re.search(r'AUC.*?([0-9]\.[0-9]{4})', line, re.IGNORECASE)
                
                if train_loss_match:
                    epochs.append(current_epoch)
                    train_loss.append(float(train_loss_match.group(1)))
                    val_loss.append(float(val_loss_match.group(1)) if val_loss_match else None)
                    val_auc.append(float(auc_match.group(1)) if auc_match else None)
        
        if len(epochs) > 0:
            return {
                'epochs': epochs,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_auc': val_auc
            }
        return None
    except:
        return None

# Parse curves for top 10 models
if len(df_results) > 0:
    top_10_archs = df_results.groupby('architecture')['auc'].mean().nlargest(10).index
    
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Validation AUC Over Epochs (Top 10 Models)', 
                       'Training Loss Over Epochs (Top 10 Models)'),
        vertical_spacing=0.12
    )
    
    colors = px.colors.qualitative.Set3
    
    for idx, arch in enumerate(top_10_archs):
        # Get fold 0 log for this architecture
        arch_logs = df_results[(df_results['architecture'] == arch) & (df_results['fold'] == 0)]
        if len(arch_logs) > 0:
            log_file = LOGS_DIR / arch_logs.iloc[0]['log_file']
            curve = parse_training_curve(log_file)
            
            if curve and len(curve['epochs']) > 0:
                color = colors[idx % len(colors)]
                
                # AUC curve
                valid_auc = [(e, a) for e, a in zip(curve['epochs'], curve['val_auc']) if a is not None]
                if valid_auc:
                    epochs_auc, auc_vals = zip(*valid_auc)
                    fig.add_trace(
                        go.Scatter(x=epochs_auc, y=auc_vals, mode='lines+markers',
                                  name=arch, line=dict(color=color, width=2),
                                  marker=dict(size=6)),
                        row=1, col=1
                    )
                
                # Loss curve
                fig.add_trace(
                    go.Scatter(x=curve['epochs'], y=curve['train_loss'], mode='lines',
                              name=arch, line=dict(color=color, width=2), showlegend=False),
                    row=2, col=1
                )
    
    fig.update_xaxes(title_text="Epoch", row=1, col=1)
    fig.update_xaxes(title_text="Epoch", row=2, col=1)
    fig.update_yaxes(title_text="Validation AUC", row=1, col=1)
    fig.update_yaxes(title_text="Training Loss", row=2, col=1)
    
    fig.update_layout(height=1000, font=dict(size=11), 
                     title_text="Training Convergence: Top 10 Architectures")
    fig.show()
    
    print("\nNote: Curves shown for fold 0 only. Full cross-fold analysis in section 6.")


Note: Curves shown for fold 0 only. Full cross-fold analysis in section 6.


## 4. Model Size vs Performance

**CRITICAL ANALYSIS**: Demonstrating the inverse relationship between parameters and performance.

In [6]:
# Map architectures to parameter counts
param_counts = {
    'seresnet10': 5.0, 'seresnet14': 8.0, 'seresnet18': 11.7, 'seresnet34': 21.8,
    'seresnet50': 28.1, 'seresnet101': 49.3,
    'resnet18': 11.2, 'resnet34': 21.3, 'resnet50': 25.6,
    'densenet121': 8.0, 'densenet169': 14.1,
    'efficientnet_b0': 5.3, 'efficientnet_b2': 9.1, 'efficientnet_b3': 12.2,
    'efficientnet_b4': 19.3, 'efficientnet_b7': 66.3,
    'mobilenetv4': 4.5,
    'vit_base': 86.6, 'vit_large': 304.3,
    'swin_tiny': 28.3,
    'convnext_tiny': 28.6, 'convnext_large': 197.8
}

if len(df_results) > 0:
    # Add parameter counts
    df_results['params_m'] = df_results['architecture'].map(
        lambda x: param_counts.get(x.lower().replace('-', '_'), np.nan)
    )
    
    # Filter valid data
    df_valid = df_results.dropna(subset=['params_m']).copy()
    
    # Create scatter plot
    fig = px.scatter(
        df_valid,
        x='params_m',
        y='auc',
        color='family',
        size='auc',
        hover_data=['architecture', 'fold'],
        title='Model Parameters vs Performance: The Smaller-is-Better Phenomenon',
        labels={'params_m': 'Parameters (Millions)', 'auc': 'Validation AUC'},
        color_discrete_sequence=px.colors.qualitative.Set2,
        height=700
    )
    
    # Add regression line
    from sklearn.linear_model import LinearRegression
    X = df_valid['params_m'].values.reshape(-1, 1)
    y = df_valid['auc'].values
    reg = LinearRegression().fit(X, y)
    
    x_trend = np.linspace(df_valid['params_m'].min(), df_valid['params_m'].max(), 100)
    y_trend = reg.predict(x_trend.reshape(-1, 1))
    
    fig.add_trace(go.Scatter(
        x=x_trend, y=y_trend, mode='lines',
        name=f'Trend (slope={reg.coef_[0]:.6f})',
        line=dict(color='red', dash='dash', width=3)
    ))
    
    # Add annotation
    corr, pval = stats.pearsonr(df_valid['params_m'], df_valid['auc'])
    fig.add_annotation(
        x=df_valid['params_m'].max() * 0.6,
        y=df_valid['auc'].min() + 0.05,
        text=f"<b>NEGATIVE CORRELATION</b><br>r={corr:.3f}, p={pval:.2e}<br>Every 10M params = {reg.coef_[0]*10:.4f} AUC decrease",
        showarrow=True,
        arrowhead=2,
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="red",
        borderwidth=2,
        font=dict(size=12, color="red")
    )
    
    fig.update_layout(font=dict(size=12))
    fig.show()
    
    print("\n=== Statistical Analysis ===")
    print(f"Pearson correlation: {corr:.4f}")
    print(f"P-value: {pval:.4e}")
    print(f"R-squared: {reg.score(X, y):.4f}")
    print(f"Interpretation: {reg.coef_[0]*10:.4f} AUC decrease per 10M parameters")
    print(f"\nThis NEGATIVE correlation is statistically significant and counterintuitive.")
    print(f"Root cause: Overfitting on limited data (~4,000 samples).")


=== Statistical Analysis ===
Pearson correlation: -0.3934
P-value: 5.6600e-06
R-squared: 0.1548
Interpretation: -0.0353 AUC decrease per 10M parameters

This NEGATIVE correlation is statistically significant and counterintuitive.
Root cause: Overfitting on limited data (~4,000 samples).


## 5. Export Results

Save all analysis results for documentation and further analysis.

In [7]:
# Export results to CSV and JSON
output_dir = GITHUB_DIR / 'notebooks' / 'outputs'
output_dir.mkdir(exist_ok=True)

if len(df_results) > 0:
    # Save complete results
    df_results.to_csv(output_dir / '04_all_training_results.csv', index=False)
    print(f"All results saved to: {output_dir / '04_all_training_results.csv'}")
    
    # Save summary statistics
    summary_stats = df_results.groupby('architecture')['auc'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).sort_values('mean', ascending=False)
    summary_stats.to_csv(output_dir / '04_architecture_summary.csv')
    print(f"Summary statistics saved to: {output_dir / '04_architecture_summary.csv'}")
    
    # Save top 10 models
    top_10 = df_results.nlargest(10, 'auc')[['architecture', 'fold', 'auc', 'log_file']]
    top_10.to_csv(output_dir / '04_top10_models.csv', index=False)
    print(f"Top 10 models saved to: {output_dir / '04_top10_models.csv'}")
    
    print("\nAll training results exported successfully!")
else:
    print("No results to export")

All results saved to: /mnt/raid0/kaggle_rsna_full/rsna_github/notebooks/outputs/04_all_training_results.csv
Summary statistics saved to: /mnt/raid0/kaggle_rsna_full/rsna_github/notebooks/outputs/04_architecture_summary.csv
Top 10 models saved to: /mnt/raid0/kaggle_rsna_full/rsna_github/notebooks/outputs/04_top10_models.csv

All training results exported successfully!
